In [10]:
import os
import pandas as pd
import torch
from torch_geometric.data import HeteroData

def load_hoddi_v2(base_path=r"C:\Users\Manav\OneDrive\Desktop\MIT\3rd Year\Sem V\ANN\Project\HODDI\Data"):
    data = HeteroData()

    # --- Paths ---
    dict_path = os.path.join(base_path, "dictionary")
    hgnn_path = os.path.join(base_path, "HGNN")

    # --- Load node dictionaries ---
    # All are .txt files, probably tab-separated — so use sep=","
    drug_df = pd.read_csv(os.path.join(dict_path, "unique_drugbank_ids.txt"), sep=",")
    sideeffect_df = pd.read_csv(os.path.join(dict_path, "Side_effects_unique.txt"), sep=",")

    # Handle flexible column names
    drug_col = "DrugBank_ID" if "DrugBank_ID" in drug_df.columns else drug_df.columns[0]
    se_col = "SideEffect" if "SideEffect" in sideeffect_df.columns else sideeffect_df.columns[0]

    drug2idx = {d: i for i, d in enumerate(drug_df[drug_col])}
    se2idx = {s: i for i, s in enumerate(sideeffect_df[se_col])}

    # --- Initialize random features ---
    data["drug"].x = torch.randn(len(drug_df), 128)
    data["sideeffect"].x = torch.randn(len(sideeffect_df), 64)

    # --- Load positive drug–drug interactions ---
    pos_dir = os.path.join(hgnn_path, "positive_samples")
    ddi_src, ddi_dst = [], []
    for file in os.listdir(pos_dir):
        if not file.endswith(".txt"):
            continue
        df = pd.read_csv(os.path.join(pos_dir, file), sep=",")
        # Column names flexible: assume first 2 are drug IDs
        c1, c2 = df.columns[:2]
        for _, row in df.iterrows():
            d1, d2 = row[c1], row[c2]
            if d1 in drug2idx and d2 in drug2idx:
                ddi_src.append(drug2idx[d1])
                ddi_dst.append(drug2idx[d2])
    if len(ddi_src) > 0:
        data["drug", "interacts", "drug"].edge_index = torch.tensor([ddi_src, ddi_dst], dtype=torch.long)

    # --- Load drug–sideeffect relations ---
    dse_path = os.path.join(dict_path, "Primkg_indication.txt")
    if os.path.exists(dse_path):
        dse_df = pd.read_csv(dse_path, sep=",")
        dse_src, dse_dst = [], []
        c1, c2 = dse_df.columns[:2]
        for _, row in dse_df.iterrows():
            d, se = row[c1], row[c2]
            if d in drug2idx and se in se2idx:
                dse_src.append(drug2idx[d])
                dse_dst.append(se2idx[se])
        if len(dse_src) > 0:
            data["drug", "causes", "sideeffect"].edge_index = torch.tensor([dse_src, dse_dst], dtype=torch.long)

    print("✅ HODDI Graph Loaded:")
    print(data)
    return data

if __name__ == "__main__":
    hoddi_graph = load_hoddi_v2()
    torch.save(hoddi_graph, "hoddi_v2_graph.pt")
    print("💾 Saved to hoddi_v2_graph.pt")


✅ HODDI Graph Loaded:
HeteroData(
  drug={ x=[3003, 128] },
  sideeffect={ x=[7350, 64] }
)
💾 Saved to hoddi_v2_graph.pt
